In [ ]:
!pip install -q ultralytics

In [ ]:
!git clone https://github.com/saikisri97/Optimisation.git
!wget -O /content/parcel_defect_trained_mac.pt https://github.com/saikisri97/Optimisation/raw/master/For_AI_Lecture/data/parcel_defect_trained_mac.pt

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n-obb.pt')

In [ ]:
model.train(
    data='/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml',
    epochs=1,
    imgsz=640,
    batch=8,
    name='parcel_defect_colab_demo',
    workers=2
)

In [ ]:
model = YOLO('/content/parcel_defect_trained_mac.pt')

In [ ]:
metrics = model.val(
    data='/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/data.yaml',
    split='test'
)
print(metrics)

In [ ]:
import os
from glob import glob

pred_dir = "runs/obb/predict"
images = glob(f"{pred_dir}/*.jpg")

print(f"🖼️ Total predicted images: {len(images)}")
print("📂 Sample:", images[:3])

In [ ]:
results = model.predict(
    source='/content/Optimisation/For_AI_Lecture/data/Parcels-with-Defect-1/test/images',
    save=True,
    imgsz=640,
    conf=0.05
)

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

predicted_dir = Path('runs/obb/predict')
class_0_img, class_1_img = None, None

for r in results:
    if r.boxes is not None and len(r.boxes) > 0:
        labels = r.boxes.cls.int().tolist()
        image_path = Path(r.path).name
        saved_img = predicted_dir / image_path
        if saved_img.exists():
            if 0 in labels and class_0_img is None:
                class_0_img = saved_img
            if 1 in labels and class_1_img is None:
                class_1_img = saved_img
        if class_0_img and class_1_img:
            break

plt.figure(figsize=(10, 5))
if class_0_img:
    img = cv2.imread(str(class_0_img))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title('Defect Parcel')
    plt.axis('off')
if class_1_img:
    img = cv2.imread(str(class_1_img))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(1, 2, 2)
    plt.imshow(img)
    plt.title('No Defect Parcel')
    plt.axis('off')
plt.suptitle('YOLOv8-OBB: One Prediction Per Class')
plt.tight_layout()
plt.show()